# 03 · Panel 2020 → 2025 y matriz de éxito

**Objetivo:** saber qué le pasó a cada restaurante entre las dos ediciones con Censo Económico (04/2020 y 05/2025) y resumirlo en una matriz **hexágono × tipo de restaurante** para el modelo de recomendación.

A cada restaurante que desaparece se le busca un **sucesor** en el mismo lugar:

| Categoría | Regla |
|---|---|
| **Continúa** | Mismo `id` y mismo negocio en 2025 |
| **Continúa (reclasificado)** | Mismo `id` y mismo negocio, pero el INEGI le cambió el código a uno que no es restaurante |
| **Continúa (cambió de nombre)** | Mismo `id`, pero los nombres no alcanzan para decidir (uno es genérico, como "TACOS DE PASTOR") |
| **Continúa (id nuevo)** | Sin `id`, pero en el mismo lugar hay un registro nuevo con nombre casi igual |
| **Reemplazo, mismo tipo / otro restaurante / otro giro** | Salió, y en su local hay otro negocio. Incluye los casos en que **el INEGI reutilizó el `id`** del local para el negocio nuevo |
| **Reemplazo incierto** | Hay un negocio nuevo en la misma dirección, pero es una plaza y no se puede saber si es el mismo local |
| **Salida sin sucesor** | Salió y no hay ningún negocio nuevo en ese lugar |

**"Mismo lugar":** misma dirección (calle y número), o a ≤ 10 m si las direcciones no se contradicen (misma calle con otro número = otro lugar). En **plazas** (3 o más negocios en la misma dirección) se exige que coincida el número interior.

**Ojo:** el INEGI a veces conserva el `id` del local cuando entra otro negocio. Por eso "mismo `id`" no basta: también se comparan los nombres.

Los comedores escolares, comunitarios y de empresa se excluyen: no son restaurantes abiertos al público.

## 0. Librerías y rutas

In [ ]:
# Requiere: conda install -n tesis-qro -c conda-forge rapidfuzz openpyxl
from pathlib import Path
import re, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h3
from rapidfuzz import fuzz
import mapas

ROOT = Path.cwd().parent
PROC, FIG = ROOT / "data/processed", ROOT / "figures"
VAL = ROOT / "data/validacion"
VAL.mkdir(parents=True, exist_ok=True)

## 1. Configuración

In [ ]:
BASE, FIN, CONTROL = "2020-04", "2025-05", "2026-05"
RES = 8                       # hexágono de la matriz (~0.74 km²)
RADIO_M = 10                  # distancia máxima para "mismo lugar"
UMBRAL_NOMBRE = 80            # similitud (0–100) para "mismo negocio"
MIN_PLAZA = 3                 # negocios en una misma dirección para considerarla plaza

TIPOS_RESTAURANTE = {
    "722511": "A la carta / comida corrida", "722512": "Mariscos", "722513": "Antojitos",
    "722514": "Tacos y tortas", "722515": "Cafeterías y neverías", "722516": "Autoservicio",
    "722517": "Pizzas, hamburguesas y pollos", "722518": "Otros para llevar",
    "722519": "Otros de consumo inmediato", "722411": "Bares y centros nocturnos",
    "722412": "Bares y centros nocturnos",
}
NO_PUBLICO = r"COMEDOR (ESCOLAR|COMUNITARIO|INDUSTRIAL|INSTITUCIONAL|DE EMPLEADOS|EMPRESARIAL)|DESAYUNADOR|COCINA COMUNITARIA"

# Palabras que no identifican a un negocio: un nombre hecho solo de ellas es "genérico"
GENERICAS = set((
    "SIN NOMBRE SN TACOS TAQUERIA COCINA ECONOMICA FONDA RESTAURANTE RESTAURANT CAFETERIA CAFE TORTAS TORTERIA "
    "ANTOJITOS POLLOS ASADOS ROSTIZADOS PALETERIA NEVERIA BAR LONCHERIA MARISCOS MARISQUERIA PIZZAS PIZZERIA GORDITAS "
    "BARBACOA CARNITAS COMIDA CORRIDA JUGOS LICUADOS HAMBURGUESAS ELOTES ESQUITES TAMALES HOT DOGS DOGOS QUESADILLAS "
    "PUESTO LOCAL MICHOACANA VENTA MEXICANA MEXICANAS MEXICANO MEXICANOS CASERA CASERAS FAMILIAR RICO RICA RICOS "
    "SABOR ANTOJERIA CENADURIA DESAYUNOS COMIDAS SERVICIO PREPARACION ALIMENTOS BEBIDAS EXPRESS"
).split())

## 2. Cargar las ediciones

- `base`: restaurantes de 2020, a los que se les sigue la pista.
- `fin`: **todo** el DENUE de 2025, porque el sucesor puede ser cualquier negocio.

In [ ]:
cols = ["id", "nom_estab", "raz_social", "codigo_act", "nom_vial", "numero_ext", "numero_int",
        "latitud", "longitud", "fecha_alta", "mun_nombre"]

def cargar(edicion):
    df = pd.read_parquet(PROC / f"denue_zmq_{edicion}.parquet")
    return df[[c for c in cols if c in df.columns]].reset_index(drop=True)

base_todo, fin = cargar(BASE), cargar(FIN)
no_pub_b = base_todo["nom_estab"].fillna("").str.upper().str.contains(NO_PUBLICO)
no_pub_f = fin["nom_estab"].fillna("").str.upper().str.contains(NO_PUBLICO)

base = base_todo[base_todo["codigo_act"].isin(TIPOS_RESTAURANTE) & ~no_pub_b].copy().reset_index(drop=True)
base["tipo"] = base["codigo_act"].map(TIPOS_RESTAURANTE)
fin["tipo"] = fin["codigo_act"].map(TIPOS_RESTAURANTE).where(~no_pub_f)     # NaN si no es restaurante público
fin["es_rest"] = fin["tipo"].notna()
print(f"Restaurantes {BASE}: {len(base):,} (excluidos {int((base_todo['codigo_act'].isin(TIPOS_RESTAURANTE) & no_pub_b).sum())} no abiertos al público)")
print(f"Negocios {FIN}: {len(fin):,} (restaurantes: {fin['es_rest'].sum():,})")

## 3. Normalización y similitud

- **Nombres:** sin acentos ni signos, sin "S.A. de C.V." ni artículos, y sin "SIN NOMBRE".
- **Similitud:** el máximo entre la comparación directa (`ratio`, tolera errores de captura como VIRRIA/BIRRIA) y la comparación con palabras ordenadas (`token_sort_ratio`, tolera otro orden).
- **Mismo negocio** (`comparar_nombres`): similitud ≥ 80, o las palabras distintivas de un nombre están contenidas en el otro ("CUCCITINI" ⊂ "FUENTE DE SODAS CUCCITINI"), o similitud ≥ 60 con la misma razón social.
- **Distinto:** ambos nombres tienen palabras distintivas, no comparten ninguna y la similitud es < 60.
- **Dudoso:** todo lo demás (por ejemplo, un nombre genérico como "TACOS DE PASTOR").
- **Vacíos:** con pandas 3 los vacíos en columnas de texto son `NaN`, no `None`; la función `_v` los trata igual en cualquier versión.

In [ ]:
def _v(x):
    # None si el valor está vacío o es NaN
    return None if x is None or (not isinstance(x, str) and pd.isna(x)) or str(x).strip() == "" else x

def _limpio(s):
    s = _v(s)
    if s is None or str(s).strip().lower() in ("nan", "none"):
        return ""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().upper()
    return re.sub(r"[^A-Z0-9 ]", " ", s)

VACIAS = r"\b(SA|DE|CV|S|A|C|V|RL|SC|SAPI|SAS|LA|EL|LOS|LAS|Y|DEL)\b"

def nombre_norm(s):
    s = re.sub(r"\bSIN NOMBRE\b|\bS N\b", " ", _limpio(s))
    return " ".join(re.sub(VACIAS, " ", s).split())

def es_generico(n):
    return (not n) or set(n.split()) <= GENERICAS

def sim(a, b):
    if not a or not b:
        return 0.0
    return max(fuzz.ratio(a, b), fuzz.token_sort_ratio(a, b))

def distintivos(n):
    # palabras que sí identifican al negocio (no genéricas, 3+ letras, no números)
    return {t for t in n.split() if t not in GENERICAS and len(t) >= 3 and not t.isdigit()}

def comparar_nombres(a, b, raz_a="", raz_b=""):
    # "mismo", "distinto" o "dudoso"
    s = sim(a, b)
    ta, tb = distintivos(a), distintivos(b)
    contenido = bool(ta) and bool(tb) and (ta <= tb or tb <= ta)      # CUCCITINI ⊂ FUENTE DE SODAS CUCCITINI
    raz = len(raz_a) >= 4 and len(raz_b) >= 4 and sim(raz_a, raz_b) >= 90
    if s >= UMBRAL_NOMBRE or contenido or (raz and s >= 60):
        return "mismo", s
    if ta and tb and not (ta & tb) and s < 60 and not raz:
        return "distinto", s
    return "dudoso", s

def direccion(calle, ext, inte):
    calle = " ".join(_limpio(calle).split())
    ext = re.sub(r"\D", "", _limpio(ext))
    inte = re.sub(r"[^A-Z0-9]", "", _limpio(inte))
    if not calle or calle in ("NINGUNO", "SIN NOMBRE") or not ext or ext == "0":
        return None, None, None
    return f"{calle}#{ext}", calle, (inte or None)

def distancia_m(la1, lo1, la2, lo2):
    return 6_371_000 * np.hypot(np.radians(la2 - la1), np.radians(lo2 - lo1) * np.cos(np.radians(la1)))

for df in (base, fin):
    df["nom_n"] = df["nom_estab"].map(nombre_norm)
    df["raz_n"] = df["raz_social"].map(nombre_norm) if "raz_social" in df else ""
    trip = [direccion(c, e, i) for c, e, i in zip(df["nom_vial"], df["numero_ext"], df["numero_int"])]
    df["dir"], df["calle"], df["dir_int"] = zip(*trip)
    df["h12"] = [h3.latlng_to_cell(la, lo, 12) for la, lo in zip(df["latitud"], df["longitud"])]

## 4. Clasificar a cada restaurante de 2020

`candidatos()` busca negocios **nuevos** de 2025 (ids que no existían en 2020) en el mismo lugar y marca si el enlace es **seguro** o **incierto** (plaza sin número interior que coincida). `clasificar()` aplica las reglas; recibe el radio como parámetro para poder probar la sensibilidad en la sección 8.

In [ ]:
ids_fin = set(fin["id"])
fin_por_id = fin.set_index("id")
N = fin[~fin["id"].isin(set(base_todo["id"]))].reset_index(drop=True)          # negocios nuevos de 2025
idx_dir = N.dropna(subset=["dir"]).groupby("dir").indices
idx_h12 = N.groupby("h12").indices
negocios_por_dir = pd.concat([fin["dir"], base_todo.assign(
    dir=[direccion(c, e, i)[0] for c, e, i in zip(base_todo["nom_vial"], base_todo["numero_ext"], base_todo["numero_int"])])["dir"]]
).dropna().value_counts()

def es_plaza(d):
    return d is not None and negocios_por_dir.get(d, 0) >= MIN_PLAZA

def candidatos(r, radio):
    r_dir, r_calle, r_int = _v(r.dir), _v(r.calle), _v(r.dir_int)
    cand = set(idx_dir.get(r_dir, [])) if r_dir else set()
    for c in h3.grid_disk(r.h12, 1 if radio <= 12 else 2):
        cand |= set(idx_h12.get(c, []))
    out = []
    for i in cand:
        n_dir, n_calle, n_int = _v(N.at[i, "dir"]), _v(N.at[i, "calle"]), _v(N.at[i, "dir_int"])
        mismo_dir = r_dir is not None and n_dir == r_dir
        if not mismo_dir:
            if distancia_m(r.latitud, r.longitud, N.at[i, "latitud"], N.at[i, "longitud"]) > radio:
                continue
            if r_dir and n_dir and r_calle == n_calle:
                continue                                  # misma calle, otro número: otro lugar
        if r_int and n_int and r_int != n_int:
            continue                                      # otro local de la misma plaza
        plaza = es_plaza(r_dir) or es_plaza(n_dir)
        seguro = (not plaza) or (r_int is not None and r_int == n_int)
        out.append((i, seguro, mismo_dir))
    return out

def mismo_negocio(r, i, mismo_dir):
    res, s_nom = comparar_nombres(r.nom_n, N.at[i, "nom_n"], _v(r.raz_n) or "", _v(N.at[i, "raz_n"]) or "")
    ok = res == "mismo"
    if es_generico(r.nom_n) or es_generico(N.at[i, "nom_n"]):
        ok = ok and mismo_dir and s_nom >= 90
    return ok, s_nom

def tipo_reemplazo(tipo_original, tipo_nuevo, es_rest_nuevo):
    if tipo_nuevo == tipo_original:
        return "Reemplazo, mismo tipo"
    return "Reemplazo, otro restaurante" if es_rest_nuevo else "Reemplazo, otro giro"

def clasificar(radio=RADIO_M):
    est, suc, simil, reut = [], [], [], []
    usados = set()
    for r in base.itertuples():
        if r.id in ids_fin:                                           # mismo id: ¿mismo negocio?
            f = fin_por_id.loc[r.id]
            res, s_nom = comparar_nombres(r.nom_n, f["nom_n"], _v(r.raz_n) or "", _v(f["raz_n"]) or "")
            if res == "distinto":                                     # el INEGI reutilizó el id
                e = tipo_reemplazo(r.tipo, f["tipo"], f["es_rest"]); reut.append(True)
            elif f["es_rest"]:
                e = "Continúa" if res == "mismo" else "Continúa (cambió de nombre)"; reut.append(False)
            else:
                e = "Continúa (reclasificado)" if res == "mismo" else "Continúa (cambió de nombre)"; reut.append(False)
            est.append(e); suc.append(r.id); simil.append(s_nom)
            continue
        cand = candidatos(r, radio)
        if not cand:
            est.append("Salida sin sucesor"); suc.append(None); simil.append(np.nan); reut.append(False)
            continue
        evals = {i: mismo_negocio(r, i, md) for i, _, md in cand}
        mejor = max(evals, key=lambda i: evals[i][1])
        if evals[mejor][0] and mejor not in usados:
            usados.add(mejor)
            est.append("Continúa (id nuevo)"); suc.append(N.at[mejor, "id"]); simil.append(evals[mejor][1]); reut.append(False)
            continue
        seguros = [i for i, s, _ in cand if s]
        if not seguros:
            i = cand[0][0]
            est.append("Reemplazo incierto"); suc.append(N.at[i, "id"]); simil.append(evals[i][1]); reut.append(False)
            continue
        mismo_t = [i for i in seguros if N.at[i, "tipo"] == r.tipo]
        otro_r = [i for i in seguros if N.at[i, "es_rest"]]
        grupo, e = ((mismo_t, "Reemplazo, mismo tipo") if mismo_t else
                    (otro_r, "Reemplazo, otro restaurante") if otro_r else
                    (seguros, "Reemplazo, otro giro"))
        i = min(grupo, key=lambda j: distancia_m(r.latitud, r.longitud, N.at[j, "latitud"], N.at[j, "longitud"]))
        est.append(e); suc.append(N.at[i, "id"]); simil.append(evals[i][1]); reut.append(False)   # el sucesor que define la categoría
    return pd.DataFrame({"estado": est, "sucesor_id": suc, "similitud": simil, "id_reutilizado": reut}, index=base.index)

ORDEN = ["Continúa", "Continúa (reclasificado)", "Continúa (cambió de nombre)", "Continúa (id nuevo)",
         "Reemplazo, mismo tipo", "Reemplazo, otro restaurante", "Reemplazo, otro giro", "Reemplazo incierto",
         "Salida sin sucesor"]
CONTINUA = ["Continúa", "Continúa (reclasificado)", "Continúa (cambió de nombre)", "Continúa (id nuevo)"]

base = base.join(clasificar(RADIO_M))
ids_reid = set(base.loc[base["estado"] == "Continúa (id nuevo)", "sucesor_id"])

tab = base["estado"].value_counts().reindex(ORDEN, fill_value=0).to_frame("restaurantes")
tab["%"] = (tab["restaurantes"] / tab["restaurantes"].sum() * 100).round(1)
print(f"Siguen en {FIN}: {base['estado'].isin(CONTINUA).mean():.1%}")
print(f"Reemplazos con id reutilizado por el INEGI: {base['id_reutilizado'].sum():,}")
tab

## 5. Entradas: restaurantes de 2025 que no estaban en 2020

Se excluyen los que en realidad son un restaurante de 2020 con `id` nuevo. Cada entrada se marca si ocupa un local que dejó un restaurante (**en local de restaurante**).

Cautela: el Censo Económico 2024 también incorpora negocios que ya existían pero no estaban en el directorio, y `fecha_alta` no lo resuelve (es la fecha en que entró al DENUE, no la de apertura).

In [ ]:
ent = fin[fin["es_rest"] & ~fin["id"].isin(set(base_todo["id"])) & ~fin["id"].isin(ids_reid)].copy()
suc_reemplazo = set(base.loc[base["estado"].str.startswith("Reemplazo"), "sucesor_id"].dropna())
ent["en_local_de_restaurante"] = ent["id"].isin(suc_reemplazo)
# Negocios que en 2020 existían pero no eran restaurante y en 2025 sí: también son entradas al giro
conv = fin[fin["es_rest"] & fin["id"].isin(set(base_todo["id"]) - set(base["id"]))].copy()
conv["en_local_de_restaurante"] = False
# Restaurantes nuevos que heredaron el id de un restaurante de 2020 (id reutilizado)
ids_reut = set(base.loc[base["id_reutilizado"], "id"])
reut_ent = fin[fin["es_rest"] & fin["id"].isin(ids_reut)].copy()
reut_ent["en_local_de_restaurante"] = True
ent = pd.concat([ent.assign(origen="nuevo"), conv.assign(origen="se convirtió en restaurante"),
                 reut_ent.assign(origen="id reutilizado")], ignore_index=True)
print(f"Entradas de restaurantes {BASE} → {FIN}: {len(ent):,}")
print(ent["origen"].value_counts().to_string())
print(f"En un local que dejó un restaurante: {ent['en_local_de_restaurante'].sum():,} ({ent['en_local_de_restaurante'].mean():.1%})")

## 6. La matriz hexágono × tipo

| Columna | Qué es |
|---|---|
| `n_base` | Restaurantes de ese tipo en 2020 |
| `continuan` | Cuántos siguen en 2025 (incluye reclasificados e id nuevo) |
| `permanencia` | `continuan / n_base` (vacío si `n_base = 0`) |
| `reemplazo_mismo_tipo` | Salieron y su local lo tomó otro del mismo tipo |
| `salidas` | Todos los que ya no están |
| `entradas` | Restaurantes nuevos de ese tipo en 2025 |
| `neto` | `entradas − salidas` |

La forma final del "éxito" se define en el notebook 04; aquí se guardan los componentes.

In [ ]:
def a_hex(df, res=RES):
    return [h3.latlng_to_cell(la, lo, res) for la, lo in zip(df["latitud"], df["longitud"])]

base["h3"], ent["h3"] = a_hex(base), a_hex(ent)
base["continua"] = base["estado"].isin(CONTINUA)
base["sale"] = ~base["continua"]
base["reemp_mismo"] = base["estado"] == "Reemplazo, mismo tipo"

m_base = base.groupby(["h3", "tipo"]).agg(n_base=("id", "size"), continuan=("continua", "sum"),
                                          salidas=("sale", "sum"), reemplazo_mismo_tipo=("reemp_mismo", "sum"))
m_ent = ent.groupby(["h3", "tipo"]).agg(entradas=("id", "size"), entradas_en_local_rest=("en_local_de_restaurante", "sum"))

hex_base = pd.read_parquet(PROC / f"hex_res{RES}_base_2020.parquet")["h3"]
tipos = sorted(set(TIPOS_RESTAURANTE.values()))
celdas = pd.Index(sorted(set(hex_base) | set(m_base.index.get_level_values(0)) | set(m_ent.index.get_level_values(0))), name="h3")
matriz = (pd.DataFrame(index=pd.MultiIndex.from_product([celdas, tipos], names=["h3", "tipo"]))
            .join(m_base).join(m_ent).fillna(0).astype(int))
matriz["permanencia"] = np.where(matriz["n_base"] > 0, matriz["continuan"] / matriz["n_base"].replace(0, 1), np.nan)
matriz["neto"] = matriz["entradas"] - matriz["salidas"]

print(f"Matriz: {len(celdas):,} hexágonos × {len(tipos)} tipos = {len(matriz):,} celdas")
print(f"Celdas con restaurantes en 2020: {(matriz['n_base'] > 0).mean():.1%} · con al menos una entrada: {(matriz['entradas'] > 0).mean():.1%}")
matriz.drop(columns="permanencia").sum().astype(int)

## 7. Primeras lecturas

### 7a. Qué pasó con los restaurantes de 2020, por tipo

In [ ]:
COLORES_ESTADO = {"Continúa": "#14213D", "Continúa (reclasificado)": "#24406B", "Continúa (cambió de nombre)": "#8DB7D3",
                  "Continúa (id nuevo)": "#2E6F95", "Reemplazo, mismo tipo": "#E8A25F", "Reemplazo, otro restaurante": "#F3C89A",
                  "Reemplazo, otro giro": "#C9C2B4", "Reemplazo incierto": "#E2DDD3", "Salida sin sucesor": "#A8551A"}
por_tipo = base.groupby(["tipo", "estado"]).size().unstack(fill_value=0).reindex(columns=ORDEN, fill_value=0)
por_tipo = por_tipo.div(por_tipo.sum(axis=1), axis=0)
por_tipo = por_tipo.loc[por_tipo[CONTINUA].sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(12, 7))
por_tipo.plot(kind="barh", stacked=True, ax=ax, width=0.7, color=[COLORES_ESTADO[c] for c in ORDEN])
ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel(f"% de los restaurantes de {BASE}"); ax.set_ylabel("")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, title=f"Situación en {FIN}")
ax.set_title(f"¿Qué pasó con los restaurantes de {BASE}?", loc="left")
plt.tight_layout(); plt.savefig(FIG / "06_transiciones_por_tipo.png", dpi=200, bbox_inches="tight"); plt.show()
(por_tipo * 100).round(1)

### 7b. Adelanto de la hipótesis 2: ¿entrar es lo mismo que quedarse?

Por tipo, correlación de rangos (Spearman) entre hexágonos de la tasa de entrada y la permanencia. Si es baja, las zonas que atraen restaurantes no son las mismas donde sobreviven.

In [ ]:
h2 = matriz[matriz["n_base"] >= 3].copy()
h2["tasa_entrada"] = h2["entradas"] / h2["n_base"]
h2["r_entrada"] = h2.groupby("tipo")["tasa_entrada"].rank()
h2["r_perm"] = h2.groupby("tipo")["permanencia"].rank()
corr = pd.DataFrame({
    "hexagonos": h2.groupby("tipo").size(),
    "spearman": h2.groupby("tipo")[["r_entrada", "r_perm"]].corr().xs("r_entrada", level=1)["r_perm"],
}).sort_values("spearman")
corr.round(2)

### 7c. Mapa de entradas de restaurantes

In [ ]:
e_hex = matriz.groupby("h3")["entradas"].sum()
e_hex = e_hex[e_hex > 0]
BORDES = [1, 2, 4, 8, 16, 32, e_hex.max() + 1]
ETIQ = ["1", "2–3", "4–7", "8–15", "16–31", f"32–{e_hex.max()}"]
cmap, norm = mapas.escala_por_clases(BORDES, ["#EAF1F6", "#BFD6E6", "#8DB7D3", "#4F8DB8", "#2E6F95", "#14213D"])
fig, ax, polys = mapas.base_mapa(e_hex.index)
coll = mapas.capa_hex(ax, polys, array=e_hex.values, cmap=cmap, norm=norm)
mapas.referencias(ax)
mapas.barra_clases(fig, ax, coll, BORDES, ETIQ, "Restaurantes nuevos por hexágono")
ax.set_title(f"Entradas de restaurantes · {BASE} → {FIN}", loc="left", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "07_mapa_entradas.png", dpi=200, bbox_inches="tight"); plt.show()

## 8. Validación sin trabajo de campo

Cuatro pruebas complementarias:

1. **Sensibilidad (8a):** ¿cambian mucho los resultados con radios de 5, 10 y 20 m?
2. **Consistencia con 2026 (8b):** un restaurante que dimos por cerrado no debería reaparecer en el DENUE de 2026.
3. **Referencia oficial (8c):** nuestra supervivencia a 5 años contra la del simulador del INEGI.
4. **Revisión manual de enlaces (8d):** comparar en un Excel los dos registros de cada par, solo con los datos.

### 8a. Sensibilidad al radio

In [ ]:
sens = {}
for radio in (5, 10, 20):
    est = base["estado"] if radio == RADIO_M else clasificar(radio)["estado"]
    sens[f"{radio} m"] = est.value_counts(normalize=True).reindex(ORDEN, fill_value=0).mul(100).round(1)
sens = pd.DataFrame(sens)
sens.loc["Siguen (total)"] = sens.loc[CONTINUA].sum()
sens

### 8b. Consistencia con la edición 2026

Para cada categoría, qué % de los `id` de 2020 aparece en el DENUE de 05/2026 (sin contar los `id` que el INEGI reutilizó para otro negocio). Lo esperado: alto en "Continúa", casi cero en las salidas. Un porcentaje alto en una categoría de salida indica que el DENUE lo "perdió" en 2025 y lo recuperó después, es decir, un cierre falso.

In [ ]:
ids_control = set(pd.read_parquet(PROC / f"denue_zmq_{CONTROL}.parquet", columns=["id"])["id"])
# Cuenta el id original; para "id nuevo", el id del sucesor (que es el mismo negocio)
en_2026 = ((base["id"].isin(ids_control) & ~base["id_reutilizado"])
           | ((base["estado"] == "Continúa (id nuevo)") & base["sucesor_id"].isin(ids_control)))
cons = (base.assign(en_2026=en_2026).groupby("estado")["en_2026"]
            .agg(casos="size", reaparece="mean").reindex(ORDEN).dropna())
cons["reaparece"] = (cons["reaparece"] * 100).round(1)
cons["casos"] = cons["casos"].astype(int)
salidas = base[~base["estado"].isin(CONTINUA) & ~base["id_reutilizado"]]
print(f"Restaurantes dados por salidos que reaparecen con su id en {CONTROL}: "
      f"{salidas['id'].isin(ids_control).sum():,} de {len(salidas):,} ({salidas['id'].isin(ids_control).mean():.1%})")
cons

### 8c. Comparación con el simulador del INEGI

El INEGI publica la probabilidad de que un negocio siga vivo después de X años, por estado y sector. El simulador no tiene "restaurantes" por separado: el sector que los incluye es **Servicios privados no financieros**.

**Cómo sacar el dato (1 minuto):**

1. Entra a <https://www.inegi.org.mx/temas/simuladorevnm/>.
2. Entidad: **Querétaro** · Sector: **Servicios privados no financieros**.
3. Construye la tabla y busca la **probabilidad de sobrevivir 5 años** para un negocio de **edad 0** (o la supervivencia acumulada a los 5 años).
4. Escribe el valor abajo, como proporción (por ejemplo, 0.35).

Ojo: nuestro dato no es exactamente lo mismo. El INEGI mide negocios **recién nacidos**, mientras que nuestros restaurantes de 2020 tienen edades mezcladas (muchos llevan años abiertos, y esos sobreviven más). Por eso nuestra supervivencia **debería salir más alta** que la del INEGI. Si sale más baja, algo está mal.

In [ ]:
SUPERVIVENCIA_INEGI_5A = None        # ← escribe aquí el valor del simulador (proporción, p. ej. 0.35)

nuestra = base["estado"].isin(CONTINUA).mean()
print(f"Nuestra supervivencia 2020 → 2025 (restaurantes de todas las edades): {nuestra:.1%}")
if SUPERVIVENCIA_INEGI_5A is not None:
    print(f"INEGI, Querétaro, servicios, nacidos, 5 años: {SUPERVIVENCIA_INEGI_5A:.1%}")
    print("Consistente: la nuestra es mayor, como se espera." if nuestra >= SUPERVIVENCIA_INEGI_5A
          else "Revisar: la nuestra es menor que la de negocios recién nacidos.")

### 8d. Revisión manual de enlaces (sin Google Maps)

Se revisan solo las categorías donde hay **dos registros que comparar**. "Salida sin sucesor" no se revisa a mano (no hay con qué comparar); la validan 8b y 8c.

Para cada renglón, una sola pregunta:

- **Continúa (reclasificado / cambió de nombre / id nuevo):** ¿los dos registros son el mismo negocio?
- **Reemplazos:** ¿es el mismo local con otro negocio? (misma dirección o número, nombre distinto)

In [ ]:
REVISAR = {
    "Continúa (reclasificado)": "¿Son el mismo negocio? (1 = sí)",
    "Continúa (cambió de nombre)": "¿Son el mismo negocio? (1 = sí)",
    "Continúa (id nuevo)": "¿Son el mismo negocio? (1 = sí)",
    "Reemplazo, mismo tipo": "¿Mismo local, otro negocio? (1 = sí)",
    "Reemplazo, otro restaurante": "¿Mismo local, otro negocio? (1 = sí)",
    "Reemplazo, otro giro": "¿Mismo local, otro negocio? (1 = sí)",
}
muestra = (base[base["estado"].isin(REVISAR)]
             .sample(frac=1, random_state=42)
             .groupby("estado").head(25))
nombres_giro = (pd.read_parquet(PROC / f"denue_zmq_{FIN}.parquet", columns=["codigo_act", "nombre_act"])
                  .drop_duplicates("codigo_act").set_index("codigo_act")["nombre_act"])
suc = fin.set_index("id")[["nom_estab", "raz_social", "codigo_act", "nom_vial", "numero_ext", "numero_int"]].add_suffix("_2025")
muestra = (muestra[["id", "estado", "id_reutilizado", "similitud", "nom_estab", "raz_social", "tipo", "mun_nombre", "nom_vial",
                    "numero_ext", "numero_int", "latitud", "longitud", "sucesor_id"]]
             .merge(suc, left_on="sucesor_id", right_index=True, how="left"))
muestra["giro_2025"] = muestra["codigo_act_2025"].map(nombres_giro)
muestra["zona"] = [h3.latlng_to_cell(la, lo, 7) for la, lo in zip(muestra["latitud"], muestra["longitud"])]
muestra = muestra.sort_values(["estado", "mun_nombre", "zona"]).reset_index(drop=True)
muestra.to_csv(VAL / "muestra_enlaces.csv", index=False, encoding="utf-8-sig")
muestra["estado"].value_counts().reindex(list(REVISAR)).dropna().astype(int)

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.formatting.rule import CellIsRule
from openpyxl.utils import get_column_letter

def num(x):
    x = "" if _v(x) is None else str(x).strip()
    return "" if x in ("0", "0.0", "nan") else (x[:-2] if x.endswith(".0") else x)

def dir_txt(calle, ext, inte):
    s = " ".join(p for p in [str(calle) if _v(calle) else "", num(ext)] if p)
    return s + (f" int. {num(inte)}" if num(inte) else "")

def txt(x):
    return "" if _v(x) is None else str(x)

AZUL, CREMA = "14213D", "F7F5F0"
wb = Workbook()
ins = wb.active; ins.title = "Instrucciones"
for f in ([["Revisión de enlaces 2020 → 2025 (solo con los datos, sin salir de la compu)"], [],
           ["1. En «Revisión», compara el registro de 2020 (columnas azules) con el de 2025 (columnas naranjas)."],
           ["2. Contesta la pregunta de la columna «Pregunta» con 1 (sí) o 0 (no) en «Respuesta»."],
           ["3. Si los datos no alcanzan para decidir, déjalo vacío. No hace falta Google Maps."],
           ["4. «Resumen» calcula el % de acierto por categoría mientras avanzas."], [],
           ["Pistas"],
           ["• Mismo negocio: nombres iguales o con errores de captura (VIRRIA/BIRRIA), misma dirección."],
           ["• Mismo local: misma calle y número (y mismo interior si lo tienen). Número distinto = otro local."],
           ["• En plazas, un interior distinto es otro local."]]):
    ins.append(f)
ins["A1"].font = Font(bold=True, size=14, color=AZUL); ins["A8"].font = Font(bold=True)
ins.column_dimensions["A"].width = 110

ws = wb.create_sheet("Revisión")
COLS = [("#", 5), ("Categoría", 24), ("Pregunta", 30), ("Respuesta (1/0)", 11),
        ("2020 · Nombre", 30), ("2020 · Razón social", 26), ("2020 · Tipo", 20), ("2020 · Dirección", 28),
        ("2025 · Nombre", 30), ("2025 · Razón social", 26), ("2025 · Giro", 30), ("2025 · Dirección", 28),
        ("Similitud nombre", 10), ("Municipio", 12), ("Notas", 26), ("Mapa (opcional)", 10), ("id_2020", 12)]
ws.append([c for c, _ in COLS])
for i, (_, w) in enumerate(COLS, 1):
    ws.column_dimensions[get_column_letter(i)].width = w
for j, c in enumerate(ws[1], 1):
    color = "2E6F95" if 5 <= j <= 8 else "C2671F" if 9 <= j <= 12 else AZUL
    c.font, c.fill = Font(bold=True, color="FFFFFF"), PatternFill("solid", fgColor=color)
    c.alignment = Alignment(wrap_text=True, vertical="center")

for i, r in muestra.iterrows():
    ws.append([i + 1, r["estado"], REVISAR[r["estado"]], None,
               txt(r["nom_estab"]), txt(r["raz_social"]), r["tipo"], dir_txt(r["nom_vial"], r["numero_ext"], r["numero_int"]),
               txt(r["nom_estab_2025"]), txt(r["raz_social_2025"]), txt(r["giro_2025"]),
               dir_txt(r["nom_vial_2025"], r["numero_ext_2025"], r["numero_int_2025"]),
               int(round(r["similitud"])) if pd.notna(r["similitud"]) else None, txt(r["mun_nombre"]), None,
               "Abrir", str(r["id"])])
    n = ws.max_row
    ws.cell(n, 16).hyperlink = f"https://www.google.com/maps?q={r['latitud']},{r['longitud']}"
    ws.cell(n, 16).font = Font(color="2E6F95", underline="single")
    for c in ws[n]:
        c.alignment = Alignment(wrap_text=True, vertical="top")
        c.border = Border(bottom=Side(style="thin", color="E2DDD3"))
        if i % 2:
            c.fill = PatternFill("solid", fgColor=CREMA)
    ws.cell(n, 4).fill = PatternFill("solid", fgColor="FFF7E6")
    ws.cell(n, 4).alignment = Alignment(horizontal="center", vertical="top")

ult = ws.max_row
dv = DataValidation(type="list", formula1='"1,0"', allow_blank=True, showErrorMessage=True,
                    errorTitle="Valor no válido", error="Usa 1 (sí) o 0 (no).")
ws.add_data_validation(dv); dv.add(f"D2:D{ult}")
ws.conditional_formatting.add(f"D2:D{ult}", CellIsRule(operator="equal", formula=["1"], fill=PatternFill("solid", fgColor="CDE8D3")))
ws.conditional_formatting.add(f"D2:D{ult}", CellIsRule(operator="equal", formula=["0"], fill=PatternFill("solid", fgColor="F6D3C9")))
ws.freeze_panes, ws.auto_filter.ref = "E2", f"A1:Q{ult}"

rs = wb.create_sheet("Resumen")
rs.append(["Categoría", "Casos", "Revisados", "Correctos", "% acierto"])
for c in rs[1]:
    c.font, c.fill = Font(bold=True, color="FFFFFF"), PatternFill("solid", fgColor=AZUL)
for k in REVISAR:
    n = rs.max_row + 1
    rs.append([k, f"=COUNTIF('Revisión'!B:B,A{n})", f"=COUNTIFS('Revisión'!B:B,A{n},'Revisión'!D:D,\"<>\")",
               f"=COUNTIFS('Revisión'!B:B,A{n},'Revisión'!D:D,1)", f"=IF(C{n}=0,\"\",D{n}/C{n})"])
    rs.cell(n, 5).number_format = "0.0%"
n = rs.max_row + 1
rs.append(["Total", f"=SUM(B2:B{n-1})", f"=SUM(C2:C{n-1})", f"=SUM(D2:D{n-1})", f"=IF(C{n}=0,\"\",D{n}/C{n})"])
rs.cell(n, 5).number_format = "0.0%"
rs.column_dimensions["A"].width = 30

wb.active = 1
salida_xlsx = VAL / "revision_enlaces.xlsx"
if salida_xlsx.exists():
    print("Ya existe revision_enlaces.xlsx; no se sobrescribe para no perder lo que ya revisaste.")
else:
    wb.save(salida_xlsx)
    print(f"Excel de revisión: {salida_xlsx} ({len(muestra)} pares)")

### 8e. Leer la revisión y calcular el acierto

In [ ]:
val = pd.read_excel(VAL / "revision_enlaces.xlsx", sheet_name="Revisión")
val["correcto"] = pd.to_numeric(val["Respuesta (1/0)"], errors="coerce")
(val.groupby("Categoría")["correcto"]
    .agg(revisados="count", acierto="mean")
    .assign(acierto=lambda d: (d["acierto"] * 100).round(1)))

## 9. Guardar

In [ ]:
quitar = ["nom_n", "raz_n", "h12", "dir", "calle", "dir_int"]
base.drop(columns=quitar).to_parquet(PROC / "transiciones_restaurantes_2020_2025.parquet", index=False)
ent.drop(columns=[c for c in quitar if c in ent]).to_parquet(PROC / "entradas_restaurantes_2020_2025.parquet", index=False)
matriz.reset_index().to_parquet(PROC / f"matriz_hex{RES}_tipo_2020_2025.parquet", index=False)
sens.to_csv(VAL / "sensibilidad_radio.csv", encoding="utf-8-sig")
cons.to_csv(VAL / "consistencia_2026.csv", encoding="utf-8-sig")
print("Guardados: transiciones, entradas, matriz, sensibilidad y consistencia")